**Spatial Autocorrelation**
This script estimates saptial autocorrelation for all exports and all imports.

**Install Libraries**


In [2]:
pip install numpy pandas duckdb libpysal esda


   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 26.5 MB/s  0:00:00

   -------- ------------------------------- 1/5 [soupsieve]
   -------- ------------------------------- 1/5 [soupsieve]
   ---------------- ----------------------- 2/5 [beautifulsoup4]
   ---------------- ----------------------- 2/5 [beautifulsoup4]
   ---------------- ----------------------- 2/5 [beautifulsoup4]
   ------------------------ --------------- 3/5 [libpysal]
   ------------------------ --------------- 3/5 [libpysal]
   ------------------------ --------------- 3/5 [libpysal]
   ------------------------ --------------- 3/5 [libpysal]
   ------------------------ --------------- 3/5 [libpysal]
   ------------------------ --------------- 3/5 [libpysal]
   ------------------------ --------------- 3/5 [libpysal]
   ------------------------ --------------- 3/5 [libpysal]
   ------------------------ --------------- 3/5 [libpysal]
   -------

In [7]:
pip install numpy pandas matplotlib seaborn duckdb libpysal esda


Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install matplotlib seaborn


Note: you may need to restart the kernel to use updated packages.


In [3]:
import libpysal
import esda
import duckdb


c:\Users\ROBERTODU\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
import numpy, pandas, matplotlib, seaborn
import duckdb
import libpysal, esda


*Implementation of Moran I
General Exports

In [1]:
import os
import math
import numpy as np
import pandas as pd

# Spatial stats
from libpysal.weights import W
from esda.moran import Moran

# Optional but strongly recommended for fast Parquet reads
import duckdb


# ============================================================
# CONFIG
# ============================================================
BASE_GEO = r"C:\Python\trade\geo"
OD_PATH = os.path.join(BASE_GEO, "OD_Matrix.csv")

BASE_TRADE = r"C:\Python\trade\dataverse_files"
PREFIX = "S2"
YEAR_START, YEAR_END = 1976, 2023

TRADE_VAR = "value_final"  # authoritative trade measure
EXPORTER_COL = "exporter"
IMPORTER_COL = "importer"

OUT_DIR = os.path.join(BASE_TRADE, "outputs_moran")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = os.path.join(OUT_DIR, f"moran_global_{PREFIX}_{YEAR_START}_{YEAR_END}_normal_inference.csv")


# ============================================================
# HELPERS
# ============================================================
def log10_1p(x: np.ndarray) -> np.ndarray:
    # Safe log10(1+x) for nonnegative x
    return np.log10(1.0 + np.maximum(x, 0.0))


def norm_cdf(z: float) -> float:
    # Standard normal CDF using erf (no scipy dependency)
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))


def two_sided_p_from_z(z: float) -> float:
    return 2.0 * (1.0 - norm_cdf(abs(z)))


# ============================================================
# BUILD FIXED SPATIAL WEIGHTS W FROM OD MATRIX
# Option 1: zero-distance pairs replaced with epsilon
# ============================================================
def build_weights_from_od(od_path: str, epsilon_factor: float = 1e-3) -> tuple[W, list[str], dict]:
    od = pd.read_csv(od_path)

    required = {"origin", "destination", "distance_km"}
    missing = required - set(od.columns)
    if missing:
        raise ValueError(f"OD_Matrix.csv missing required columns: {missing}")

    # Universe and stable order (sorted, deterministic)
    codes = sorted(set(od["origin"].astype(str)) | set(od["destination"].astype(str)))
    idx = {c: i for i, c in enumerate(codes)}
    n = len(codes)

    # Build dense distance matrix
    D = np.full((n, n), np.nan, dtype=float)
    for r in od.itertuples(index=False):
        i = idx[str(r.origin)]
        j = idx[str(r.destination)]
        D[i, j] = float(r.distance_km)

    # Check symmetry (not strictly required, but expected)
    # We won't force symmetry; we just work with what is provided.

    # Identify min positive distance to define epsilon
    positive = D[(~np.isnan(D)) & (D > 0)]
    if positive.size == 0:
        raise ValueError("OD matrix has no positive distances; cannot build weights.")

    dmin = float(np.min(positive))
    eps = dmin * epsilon_factor

    # Count zero-distance off-diagonal pairs
    offdiag_zero = np.where((D == 0.0) & (~np.eye(n, dtype=bool)))
    n_zero_pairs = int(len(offdiag_zero[0]))

    # Replace zeros with epsilon (Option 1)
    D[(D == 0.0) & (~np.eye(n, dtype=bool))] = eps

    # Convert to weights: w_ij = 1/d_ij for i != j, 0 on diagonal, ignore NaNs
    W_dense = np.zeros((n, n), dtype=float)
    mask = (~np.isnan(D)) & (~np.eye(n, dtype=bool))
    W_dense[mask] = 1.0 / D[mask]

    # Row-standardize (if a row sums to 0, leave it as all zeros; Moran will fail -> we stop later)
    row_sums = W_dense.sum(axis=1)
    zero_rows = np.where(row_sums == 0.0)[0]
    if zero_rows.size > 0:
        raise ValueError(
            f"W has {zero_rows.size} rows with zero total weight (disconnected units). "
            f"First few: {[codes[i] for i in zero_rows[:10]]}"
        )
    W_dense = (W_dense.T / row_sums).T

    # Build libpysal W object
    neighbors = {}
    weights = {}
    for i, c in enumerate(codes):
        js = np.where(W_dense[i, :] > 0)[0]
        neighbors[c] = [codes[j] for j in js]
        weights[c] = [float(W_dense[i, j]) for j in js]

    w = W(neighbors, weights, silence_warnings=True)

    meta = {
        "n_units": n,
        "min_positive_distance_km": dmin,
        "epsilon_km": eps,
        "zero_distance_pairs_offdiag": n_zero_pairs,
    }
    return w, codes, meta


# ============================================================
# LOAD + AGGREGATE TRADE FOR A YEAR (exports/imports from value_final)
# ============================================================
def load_trade_vectors_for_year(parquet_path: str, universe_codes: list[str]) -> tuple[np.ndarray, np.ndarray, dict]:
    if not os.path.exists(parquet_path):
        raise FileNotFoundError(f"Missing trade file: {parquet_path}")

    # Use DuckDB for speed and to avoid loading unnecessary columns
    con = duckdb.connect(database=":memory:")

    # Exports: sum(value_final) by exporter
    q_exp = f"""
        SELECT
            CAST({EXPORTER_COL} AS VARCHAR) AS code,
            SUM(CAST({TRADE_VAR} AS DOUBLE)) AS total
        FROM read_parquet('{parquet_path}')
        GROUP BY 1
    """
    exp_df = con.execute(q_exp).df()

    # Imports: sum(value_final) by importer
    q_imp = f"""
        SELECT
            CAST({IMPORTER_COL} AS VARCHAR) AS code,
            SUM(CAST({TRADE_VAR} AS DOUBLE)) AS total
        FROM read_parquet('{parquet_path}')
        GROUP BY 1
    """
    imp_df = con.execute(q_imp).df()
    con.close()

    # Map into universe with zeros
    uni = pd.Index(universe_codes)

    exp_s = exp_df.set_index("code")["total"].reindex(uni).fillna(0.0)
    imp_s = imp_df.set_index("code")["total"].reindex(uni).fillna(0.0)

    # Diagnostics: trade codes not in universe (cannot be located)
    exp_not_loc = int((~exp_df["code"].isin(uni)).sum())
    imp_not_loc = int((~imp_df["code"].isin(uni)).sum())

    meta = {
        "exp_not_locatable_codes": exp_not_loc,
        "imp_not_locatable_codes": imp_not_loc,
        "exp_zero_share": float((exp_s.values == 0).mean()),
        "imp_zero_share": float((imp_s.values == 0).mean()),
    }

    return exp_s.values.astype(float), imp_s.values.astype(float), meta


# ============================================================
# MORAN ESTIMATION (normality assumption)
# ============================================================
def moran_normal_report(y: np.ndarray, w: W) -> dict:
    # esda Moran centers internally; we rely on its analytical moments
    m = Moran(y, w, permutations=0)

    I = float(m.I)
    EI = float(m.EI)
    VI = float(m.VI_norm)  # variance under normality
    if not np.isfinite(VI) or VI <= 0:
        raise ValueError(f"Invalid variance VI_norm={VI}. Moran inference not defined.")

    se = float(math.sqrt(VI))
    z = float((I - EI) / se)
    p = float(two_sided_p_from_z(z))
    ci_5 = float(I - 1.96 * se)
    ci_95 = float(I + 1.96 * se)

    return {
        "moran_i": I,
        "mean_H0": EI,
        "variance_H0": VI,
        "z": z,
        "p_value": p,
        "ci_5": ci_5,
        "ci_95": ci_95,
    }


# ============================================================
# MAIN
# ============================================================
def main():
    # 1) Build weights once
    w, universe, w_meta = build_weights_from_od(OD_PATH, epsilon_factor=1e-3)
    n = len(universe)

    print("Weights built.")
    print(f"  n_units={w_meta['n_units']}")
    print(f"  min_positive_distance_km={w_meta['min_positive_distance_km']:.6f}")
    print(f"  epsilon_km={w_meta['epsilon_km']:.9f}")
    print(f"  zero_distance_pairs_offdiag={w_meta['zero_distance_pairs_offdiag']}")

    rows = []

    # 2) Loop years
    for year in range(YEAR_START, YEAR_END + 1):
        parquet_path = os.path.join(BASE_TRADE, f"{PREFIX}_{year}.parquet")
        exp_raw, imp_raw, t_meta = load_trade_vectors_for_year(parquet_path, universe)

        # 3) Transform
        y_exp = log10_1p(exp_raw)
        y_imp = log10_1p(imp_raw)

        # 4) Moran + inference
        rep_exp = moran_normal_report(y_exp, w)
        rep_imp = moran_normal_report(y_imp, w)

        # 5) Append rows (exports/imports)
        rows.append({
            "year": year,
            "flow": "exports",
            "n": n,
            **rep_exp,
            **t_meta,
        })
        rows.append({
            "year": year,
            "flow": "imports",
            "n": n,
            **rep_imp,
            **t_meta,
        })

        if year % 5 == 0:
            print(f"Processed {year}")

    out = pd.DataFrame(rows)

    # Stable column order
    col_order = [
        "year", "flow", "n",
        "moran_i", "mean_H0", "variance_H0", "z", "p_value", "ci_5", "ci_95",
        "exp_zero_share", "imp_zero_share",
        "exp_not_locatable_codes", "imp_not_locatable_codes",
    ]
    out = out[col_order]

    out.to_csv(OUT_CSV, index=False)
    print(f"\nSaved: {OUT_CSV}")
    print(out.head(6).to_string(index=False))


if __name__ == "__main__":
    main()


c:\Users\ROBERTODU\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Weights built.
  n_units=252
  min_positive_distance_km=19.857463
  epsilon_km=0.019857463
  zero_distance_pairs_offdiag=28
Processed 1980
Processed 1985
Processed 1990
Processed 1995
Processed 2000
Processed 2005
Processed 2010
Processed 2015
Processed 2020

Saved: C:\Python\trade\dataverse_files\outputs_moran\moran_global_S2_1976_2023_normal_inference.csv
 year    flow   n  moran_i   mean_H0  variance_H0        z  p_value      ci_5    ci_95  exp_zero_share  imp_zero_share  exp_not_locatable_codes  imp_not_locatable_codes
 1976 exports 252 0.022561 -0.003984     0.000671 1.024905 0.305408 -0.028204 0.073327        0.226190        0.218254                        0                        0
 1976 imports 252 0.019262 -0.003984     0.000671 0.897533 0.369435 -0.031503 0.070028        0.226190        0.218254                        0                        0
 1977 exports 252 0.020889 -0.003984     0.000671 0.960316 0.336896 -0.029876 0.071654        0.230159        0.214286               

Graph

In [4]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------------------------------------------
# CONFIG: point to your CSV (the one produced by the global Moran script)
# -------------------------------------------------------------------
CSV_PATH = r"C:\Python\trade\dataverse_files\outputs_moran\moran_global_S2_1976_2023_normal_inference.csv"

# Output directory = same directory as the CSV
OUT_DIR = os.path.dirname(CSV_PATH)

# Output files
OUT_EXPORTS = os.path.join(OUT_DIR, "moran_global_exports.png")
OUT_IMPORTS = os.path.join(OUT_DIR, "moran_global_imports.png")

# -------------------------------------------------------------------
# LOAD + BASIC VALIDATION
# -------------------------------------------------------------------
df = pd.read_csv(CSV_PATH)

required_cols = {"year", "flow", "moran_i", "ci_5", "ci_95"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"CSV is missing required columns: {missing}")

# Ensure numeric years and sort
df["year"] = pd.to_numeric(df["year"], errors="raise")
df = df.sort_values(["flow", "year"]).reset_index(drop=True)

# -------------------------------------------------------------------
# PLOT FUNCTION (no title by request)
# -------------------------------------------------------------------
def plot_flow(flow_name: str, out_path: str):
    d = df[df["flow"].str.lower() == flow_name.lower()].copy()
    if d.empty:
        raise ValueError(f"No rows found for flow='{flow_name}'. Check 'flow' column values.")

    x = d["year"].values
    y = d["moran_i"].values
    lo = d["ci_5"].values
    hi = d["ci_95"].values

    fig, ax = plt.subplots(figsize=(11, 6), dpi=200)

    # Shaded CI band (upper/lower bounds)
    ax.fill_between(x, lo, hi, alpha=0.25)

    # Zero reference line
    ax.axhline(0.0, color='black', linewidth=1.5)

    # Moran I line
    ax.plot(x, y, linewidth=2.5)

    # Axis formatting
    ax.set_xlabel("")
    ax.set_ylabel("")

    # Keep x-ticks readable (similar to your example)
    # Show every 2 years; adjust if you prefer every 1 or 5
    ticks = x[::2]
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(int(t)) for t in ticks], rotation=90)

    # Light grid, no title
    ax.grid(True, axis="y", alpha=0.3)
    ax.grid(False, axis="x")

    plt.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)

# -------------------------------------------------------------------
# GENERATE BOTH FIGURES
# -------------------------------------------------------------------
plot_flow("exports", OUT_EXPORTS)
plot_flow("imports", OUT_IMPORTS)

print("Saved:")
print(OUT_EXPORTS)
print(OUT_IMPORTS)


Saved:
C:\Python\trade\dataverse_files\outputs_moran\moran_global_exports.png
C:\Python\trade\dataverse_files\outputs_moran\moran_global_imports.png


**Estimation of Moran I for 2 Digit SCIT**

In [5]:
import os
import math
import numpy as np
import pandas as pd

from libpysal.weights import W
from esda.moran import Moran
import duckdb


# ============================================================
# CONFIG
# ============================================================
BASE_GEO = r"C:\Python\trade\geo"
OD_PATH = os.path.join(BASE_GEO, "OD_Matrix.csv")

BASE_TRADE = r"C:\Python\trade\dataverse_files"
PREFIX = "S2"
YEAR_START, YEAR_END = 1976, 2023

TRADE_VAR = "value_final"          # authoritative trade measure
EXPORTER_COL = "exporter"
IMPORTER_COL = "importer"
COMMODITY_COL = "commoditycode"

OUT_DIR = os.path.join(BASE_TRADE, "outputs_moran")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = os.path.join(OUT_DIR, f"moran_sitc2_{PREFIX}_{YEAR_START}_{YEAR_END}_normal_inference.csv")


# ============================================================
# HELPERS
# ============================================================
def log10_1p(x: np.ndarray) -> np.ndarray:
    return np.log10(1.0 + np.maximum(x, 0.0))


def norm_cdf(z: float) -> float:
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))


def two_sided_p_from_z(z: float) -> float:
    return 2.0 * (1.0 - norm_cdf(abs(z)))


# ============================================================
# BUILD FIXED SPATIAL WEIGHTS W FROM OD MATRIX
# Option 1: zero-distance pairs replaced with epsilon
# ============================================================
def build_weights_from_od(od_path: str, epsilon_factor: float = 1e-3) -> tuple[W, list[str], dict]:
    od = pd.read_csv(od_path)

    required = {"origin", "destination", "distance_km"}
    missing = required - set(od.columns)
    if missing:
        raise ValueError(f"OD_Matrix.csv missing required columns: {missing}")

    codes = sorted(set(od["origin"].astype(str)) | set(od["destination"].astype(str)))
    idx = {c: i for i, c in enumerate(codes)}
    n = len(codes)

    D = np.full((n, n), np.nan, dtype=float)
    for r in od.itertuples(index=False):
        i = idx[str(r.origin)]
        j = idx[str(r.destination)]
        D[i, j] = float(r.distance_km)

    positive = D[(~np.isnan(D)) & (D > 0)]
    if positive.size == 0:
        raise ValueError("OD matrix has no positive distances; cannot build weights.")

    dmin = float(np.min(positive))
    eps = dmin * epsilon_factor

    offdiag_zero = np.where((D == 0.0) & (~np.eye(n, dtype=bool)))
    n_zero_pairs = int(len(offdiag_zero[0]))

    D[(D == 0.0) & (~np.eye(n, dtype=bool))] = eps

    W_dense = np.zeros((n, n), dtype=float)
    mask = (~np.isnan(D)) & (~np.eye(n, dtype=bool))
    W_dense[mask] = 1.0 / D[mask]

    row_sums = W_dense.sum(axis=1)
    zero_rows = np.where(row_sums == 0.0)[0]
    if zero_rows.size > 0:
        raise ValueError(
            f"W has {zero_rows.size} rows with zero total weight (disconnected units). "
            f"First few: {[codes[i] for i in zero_rows[:10]]}"
        )
    W_dense = (W_dense.T / row_sums).T

    neighbors = {}
    weights = {}
    for i, c in enumerate(codes):
        js = np.where(W_dense[i, :] > 0)[0]
        neighbors[c] = [codes[j] for j in js]
        weights[c] = [float(W_dense[i, j]) for j in js]

    w = W(neighbors, weights, silence_warnings=True)

    meta = {
        "n_units": n,
        "min_positive_distance_km": dmin,
        "epsilon_km": eps,
        "zero_distance_pairs_offdiag": n_zero_pairs,
    }
    return w, codes, meta


# ============================================================
# MORAN ESTIMATION (normality assumption)
# ============================================================
def moran_normal_report(y: np.ndarray, w: W) -> dict:
    m = Moran(y, w, permutations=0)

    I = float(m.I)
    EI = float(m.EI)
    VI = float(m.VI_norm)
    if not np.isfinite(VI) or VI <= 0:
        raise ValueError(f"Invalid variance VI_norm={VI}. Moran inference not defined.")

    se = float(math.sqrt(VI))
    z = float((I - EI) / se)
    p = float(two_sided_p_from_z(z))
    ci_5 = float(I - 1.96 * se)
    ci_95 = float(I + 1.96 * se)

    return {
        "moran_i": I,
        "mean_H0": EI,
        "variance_H0": VI,
        "z": z,
        "p_value": p,
        "ci_5": ci_5,
        "ci_95": ci_95,
    }


# ============================================================
# MAIN
# ============================================================
def main():
    # 1) Build weights once
    w, universe, w_meta = build_weights_from_od(OD_PATH, epsilon_factor=1e-3)
    n = len(universe)
    uni = pd.Index(universe)

    print("Weights built.")
    print(f"  n_units={w_meta['n_units']}")
    print(f"  min_positive_distance_km={w_meta['min_positive_distance_km']:.6f}")
    print(f"  epsilon_km={w_meta['epsilon_km']:.9f}")
    print(f"  zero_distance_pairs_offdiag={w_meta['zero_distance_pairs_offdiag']}")

    rows = []

    # 2) Loop years
    for year in range(YEAR_START, YEAR_END + 1):
        parquet_path = os.path.join(BASE_TRADE, f"{PREFIX}_{year}.parquet")
        if not os.path.exists(parquet_path):
            raise FileNotFoundError(f"Missing trade file: {parquet_path}")

        con = duckdb.connect(database=":memory:")

        # Aggregate exports by (exporter, sitc2)
        q_exp = f"""
            SELECT
                CAST({EXPORTER_COL} AS VARCHAR) AS code,
                SUBSTR(CAST({COMMODITY_COL} AS VARCHAR), 1, 2) AS sitc2,
                SUM(CAST({TRADE_VAR} AS DOUBLE)) AS total
            FROM read_parquet('{parquet_path}')
            GROUP BY 1, 2
        """
        exp = con.execute(q_exp).df()

        # Aggregate imports by (importer, sitc2)
        q_imp = f"""
            SELECT
                CAST({IMPORTER_COL} AS VARCHAR) AS code,
                SUBSTR(CAST({COMMODITY_COL} AS VARCHAR), 1, 2) AS sitc2,
                SUM(CAST({TRADE_VAR} AS DOUBLE)) AS total
            FROM read_parquet('{parquet_path}')
            GROUP BY 1, 2
        """
        imp = con.execute(q_imp).df()

        con.close()

        # Ensure sitc2 is string with leading zeros preserved
        exp["sitc2"] = exp["sitc2"].astype(str).str.zfill(2)
        imp["sitc2"] = imp["sitc2"].astype(str).str.zfill(2)

        # Sector universe for this year (union of exp & imp)
        sectors = sorted(set(exp["sitc2"]) | set(imp["sitc2"]))

        # Diagnostics: non-locatable country codes (cannot be placed in W)
        exp_not_loc = int((~exp["code"].isin(uni)).sum())
        imp_not_loc = int((~imp["code"].isin(uni)).sum())

        # Pre-index for fast reindexing per sector
        # (We keep them as DataFrames and filter by sitc2; acceptable speed for SITC-2.)
        for s in sectors:
            # Exports vector for sector s
            exp_s = exp.loc[exp["sitc2"] == s, ["code", "total"]]
            exp_vec = exp_s.set_index("code")["total"].reindex(uni).fillna(0.0).values.astype(float)

            # Imports vector for sector s
            imp_s = imp.loc[imp["sitc2"] == s, ["code", "total"]]
            imp_vec = imp_s.set_index("code")["total"].reindex(uni).fillna(0.0).values.astype(float)

            # Transform
            y_exp = log10_1p(exp_vec)
            y_imp = log10_1p(imp_vec)

            # Share of zeros (diagnostic)
            exp_zero_share = float((exp_vec == 0.0).mean())
            imp_zero_share = float((imp_vec == 0.0).mean())

            # Moran
            rep_exp = moran_normal_report(y_exp, w)
            rep_imp = moran_normal_report(y_imp, w)

            rows.append({
                "year": year,
                "sitc2": s,
                "flow": "exports",
                "n": n,
                **rep_exp,
                "zero_share": exp_zero_share,
                "exp_not_locatable_codes": exp_not_loc,
                "imp_not_locatable_codes": imp_not_loc,
            })
            rows.append({
                "year": year,
                "sitc2": s,
                "flow": "imports",
                "n": n,
                **rep_imp,
                "zero_share": imp_zero_share,
                "exp_not_locatable_codes": exp_not_loc,
                "imp_not_locatable_codes": imp_not_loc,
            })

        if year % 5 == 0:
            print(f"Processed {year} (sectors={len(sectors)})")

    out = pd.DataFrame(rows)

    # Stable column order
    col_order = [
        "year", "sitc2", "flow", "n",
        "moran_i", "mean_H0", "variance_H0", "z", "p_value", "ci_5", "ci_95",
        "zero_share", "exp_not_locatable_codes", "imp_not_locatable_codes",
    ]
    out = out[col_order].sort_values(["year", "sitc2", "flow"]).reset_index(drop=True)

    out.to_csv(OUT_CSV, index=False)
    print(f"\nSaved: {OUT_CSV}")
    print(out.head(10).to_string(index=False))


if __name__ == "__main__":
    main()


Weights built.
  n_units=252
  min_positive_distance_km=19.857463
  epsilon_km=0.019857463
  zero_distance_pairs_offdiag=28
Processed 1980 (sectors=70)
Processed 1985 (sectors=70)
Processed 1990 (sectors=70)
Processed 1995 (sectors=70)
Processed 2000 (sectors=70)
Processed 2005 (sectors=70)
Processed 2010 (sectors=70)
Processed 2015 (sectors=70)
Processed 2020 (sectors=70)

Saved: C:\Python\trade\dataverse_files\outputs_moran\moran_sitc2_S2_1976_2023_normal_inference.csv
 year sitc2    flow   n  moran_i   mean_H0  variance_H0        z  p_value      ci_5    ci_95  zero_share  exp_not_locatable_codes  imp_not_locatable_codes
 1976    00 exports 252 0.038639 -0.003984     0.000671 1.645641 0.099838 -0.012126 0.089404    0.464286                        0                        0
 1976    00 imports 252 0.032302 -0.003984     0.000671 1.400994 0.161216 -0.018463 0.083067    0.333333                        0                        0
 1976    01 exports 252 0.026352 -0.003984     0.000671 1.1

**Moran I Heatmap 2 Digit**

In [29]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ============================================================
# CONFIG
# ============================================================
# Moran SITC-2 results (from your estimation script)
MORAN_SITC2_CSV = r"C:\Python\trade\dataverse_files\outputs_moran\moran_sitc2_S2_1976_2023_normal_inference.csv"

# SITC-2 dictionary (code + full name + nickname)
CODES_DIR = r"C:\Python\trade\dataverse_files\S02 CODES"
SITC2_DICT_PATH = os.path.join(CODES_DIR, "sitc2-2digit.txt")  # validate contents; must be 2-digit codes

# Output directory = same directory as Moran CSV
OUT_DIR = os.path.dirname(MORAN_SITC2_CSV)

OUT_EXPORTS = os.path.join(OUT_DIR, "heatmap_moran_sitc2_level_exports.png")
OUT_IMPORTS = os.path.join(OUT_DIR, "heatmap_moran_sitc2_level_imports.png")


# ============================================================
# ROBUST TXT PARSER (delimiter-agnostic)
# ============================================================
def read_code_dictionary(path: str) -> pd.DataFrame:
    """
    Reads a code dictionary from a .txt file with unknown delimiter.
    Expected columns (in some order): code, full name, nickname.
    We infer delimiter among: tab, pipe, semicolon, comma, multiple spaces.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    # Read raw lines
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    # Drop obvious comment lines
    lines = [ln for ln in lines if not ln.startswith("#")]

    # Try candidate delimiters
    candidates = ["\t", "|", ";", ","]
    best = None

    def try_parse(delim: str):
        rows = [ln.split(delim) for ln in lines]
        k = max(len(r) for r in rows)
        if k < 2:
            return None
        # normalize row lengths
        rows = [r + [""] * (k - len(r)) for r in rows]
        df = pd.DataFrame(rows)
        return df

    # Attempt structured delimiters first
    for d in candidates:
        df = try_parse(d)
        if df is None:
            continue
        # prefer delimiters yielding at least 3 columns
        if df.shape[1] >= 3:
            best = df
            break

    # Fallback: split on 2+ spaces
    if best is None:
        rows = [re.split(r"\s{2,}", ln) for ln in lines]
        k = max(len(r) for r in rows)
        rows = [r + [""] * (k - len(r)) for r in rows]
        best = pd.DataFrame(rows)

    # Keep first 3 cols as [code, name, nickname] by default
    # If file has header row, detect and drop it.
    df = best.iloc[:, :3].copy()
    df.columns = ["code", "name", "nickname"]
    
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip()
    
    # Drop header-like first row if it contains non-code text in code col
    if len(df) and not re.fullmatch(r"\d{1,3}", str(df.loc[0, "code"]).strip() or ""):
        # common headers: code / sitc / commodity
        df = df.iloc[1:].reset_index(drop=True)

    # Force strings + preserve leading zeros
    df["code"] = df["code"].astype(str).str.strip()
    df["name"] = df["name"].astype(str).str.strip()
    df["nickname"] = df["nickname"].astype(str).str.strip()

    return df


def build_sitc2_labels(dict_df: pd.DataFrame) -> pd.DataFrame:
    # Normalize to 2-digit codes
    dict_df = dict_df.copy()
    dict_df["code"] = dict_df["code"].str.extract(r"(\d+)")[0].fillna(dict_df["code"])
    dict_df["code"] = dict_df["code"].astype(str).str.zfill(2)

    # If this file accidentally contains 3-digit codes, warn early
    lens = dict_df["code"].str.len().value_counts().to_dict()
    if set(lens.keys()) - {2}:
        # If many codes are length 3, user likely pointed to wrong file
        raise ValueError(
            f"Dictionary codes are not strictly 2-digit after zfill. Length distribution: {lens}. "
            f"Check SITC2_DICT_PATH points to the 2-digit file."
        )

    # Build display label: "00 – Nickname"
    # If nickname missing, fall back to name, then code.
    dict_df["display"] = dict_df["nickname"].replace({"": np.nan})
    dict_df["display"] = dict_df["display"].fillna(dict_df["name"].replace({"": np.nan}))
    dict_df["display"] = dict_df["display"].fillna(dict_df["code"])
    dict_df["display"] = dict_df["code"] + " – " + dict_df["display"]

    # Deduplicate by code (keep first)
    dict_df = dict_df.drop_duplicates(subset=["code"], keep="first")

    return dict_df[["code", "display"]]


# ============================================================
# HEATMAP PLOTTER
# ============================================================
def plot_sitc2_heatmap(moran_csv: str, labels_df: pd.DataFrame, flow: str, out_path: str):
    df = pd.read_csv(moran_csv, dtype={"sitc2": str, "flow": str})
    if "sitc2" not in df.columns:
        raise ValueError("Expected 'sitc2' column in Moran SITC-2 CSV.")
    if "moran_i" not in df.columns:
        raise ValueError("Expected 'moran_i' column in Moran SITC-2 CSV.")
    if "year" not in df.columns:
        raise ValueError("Expected 'year' column in Moran SITC-2 CSV.")
    if "flow" not in df.columns:
        raise ValueError("Expected 'flow' column in Moran SITC-2 CSV.")

    # Normalize sitc2 format
    df["sitc2"] = df["sitc2"].astype(str).str.zfill(2)
    df["year"] = pd.to_numeric(df["year"], errors="raise")

    d = df[df["flow"].str.lower() == flow.lower()].copy()
    if d.empty:
        raise ValueError(f"No rows found for flow='{flow}'")

    # Pivot to matrix (rows=sitc2, cols=year)
    mat = d.pivot(index="sitc2", columns="year", values="moran_i")

    # Sort axes
    mat = mat.sort_index()
    mat = mat.reindex(sorted(mat.columns), axis=1)

    # Build y labels
    label_map = dict(zip(labels_df["code"], labels_df["display"]))
    y_labels = [label_map.get(code, code) for code in mat.index]

    # Diverging around
    arr = mat.values.astype(float)
    norm = TwoSlopeNorm(vmin=-0.20, vcenter=0.0, vmax=0.20)
    
    # Plot (no title requested)
    fig, ax = plt.subplots(figsize=(18, 12), dpi=200)

    im = ax.imshow(arr, aspect="auto", interpolation="nearest", norm=norm, cmap="RdBu_r")

    # X axis (years)
    years = mat.columns.tolist()
    ax.set_xticks(range(len(years)))

    # reduce tick density (every 2 years)
    show_every = 2
    xticklabels = [str(int(y)) if (i % show_every == 0) else "" for i, y in enumerate(years)]
    ax.set_xticklabels(xticklabels, rotation=90)

    # Y axis (sectors)
    ax.set_yticks(range(len(y_labels)))
    ax.set_yticklabels(y_labels)

    ax.set_xlabel("")
    ax.set_ylabel("")

    # Colorbar
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Moran's I")

    plt.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# RUN
# ============================================================
dict_raw = read_code_dictionary(SITC2_DICT_PATH)
labels = build_sitc2_labels(dict_raw)

plot_sitc2_heatmap(MORAN_SITC2_CSV, labels, flow="exports", out_path=OUT_EXPORTS)
plot_sitc2_heatmap(MORAN_SITC2_CSV, labels, flow="imports", out_path=OUT_IMPORTS)

print("Saved:")
print(OUT_EXPORTS)
print(OUT_IMPORTS)


Saved:
C:\Python\trade\dataverse_files\outputs_moran\heatmap_moran_sitc2_level_exports.png
C:\Python\trade\dataverse_files\outputs_moran\heatmap_moran_sitc2_level_imports.png


**Moran I 3 digit**

In [6]:
import os
import math
import numpy as np
import pandas as pd

from libpysal.weights import W
from esda.moran import Moran
import duckdb


# ============================================================
# CONFIG
# ============================================================
BASE_GEO = r"C:\Python\trade\geo"
OD_PATH = os.path.join(BASE_GEO, "OD_Matrix.csv")

BASE_TRADE = r"C:\Python\trade\dataverse_files"
PREFIX = "S2"
YEAR_START, YEAR_END = 1976, 2023

TRADE_VAR = "value_final"
EXPORTER_COL = "exporter"
IMPORTER_COL = "importer"
COMMODITY_COL = "commoditycode"

OUT_DIR = os.path.join(BASE_TRADE, "outputs_moran")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = os.path.join(
    OUT_DIR, f"moran_sitc3_{PREFIX}_{YEAR_START}_{YEAR_END}_normal_inference.csv"
)


# ============================================================
# HELPERS
# ============================================================
def log10_1p(x: np.ndarray) -> np.ndarray:
    return np.log10(1.0 + np.maximum(x, 0.0))


def norm_cdf(z: float) -> float:
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))


def two_sided_p_from_z(z: float) -> float:
    return 2.0 * (1.0 - norm_cdf(abs(z)))


# ============================================================
# BUILD FIXED SPATIAL WEIGHTS W
# ============================================================
def build_weights_from_od(od_path: str, epsilon_factor: float = 1e-3) -> tuple[W, list[str]]:
    od = pd.read_csv(od_path)

    codes = sorted(set(od["origin"].astype(str)) | set(od["destination"].astype(str)))
    idx = {c: i for i, c in enumerate(codes)}
    n = len(codes)

    D = np.full((n, n), np.nan)
    for r in od.itertuples(index=False):
        D[idx[str(r.origin)], idx[str(r.destination)]] = float(r.distance_km)

    positive = D[(~np.isnan(D)) & (D > 0)]
    if positive.size == 0:
        raise ValueError("No positive distances in OD matrix.")

    eps = float(np.min(positive)) * epsilon_factor
    D[(D == 0.0) & (~np.eye(n, dtype=bool))] = eps

    W_dense = np.zeros((n, n))
    mask = (~np.isnan(D)) & (~np.eye(n, dtype=bool))
    W_dense[mask] = 1.0 / D[mask]

    row_sums = W_dense.sum(axis=1)
    if np.any(row_sums == 0):
        raise ValueError("Disconnected spatial units in W.")

    W_dense = (W_dense.T / row_sums).T

    neighbors, weights = {}, {}
    for i, c in enumerate(codes):
        js = np.where(W_dense[i] > 0)[0]
        neighbors[c] = [codes[j] for j in js]
        weights[c] = [float(W_dense[i, j]) for j in js]

    return W(neighbors, weights, silence_warnings=True), codes


# ============================================================
# MORAN (NORMALITY)
# ============================================================
def moran_normal_report(y: np.ndarray, w: W) -> dict:
    m = Moran(y, w, permutations=0)

    I = float(m.I)
    EI = float(m.EI)
    VI = float(m.VI_norm)
    if not np.isfinite(VI) or VI <= 0:
        raise ValueError("Invalid Moran variance.")

    se = math.sqrt(VI)
    z = (I - EI) / se

    return {
        "moran_i": I,
        "mean_H0": EI,
        "variance_H0": VI,
        "z": z,
        "p_value": two_sided_p_from_z(z),
        "ci_5": I - 1.96 * se,
        "ci_95": I + 1.96 * se,
    }


# ============================================================
# MAIN
# ============================================================
def main():
    w, universe = build_weights_from_od(OD_PATH)
    uni = pd.Index(universe)
    n = len(uni)

    rows = []

    for year in range(YEAR_START, YEAR_END + 1):
        parquet = os.path.join(BASE_TRADE, f"{PREFIX}_{year}.parquet")
        if not os.path.exists(parquet):
            raise FileNotFoundError(parquet)

        con = duckdb.connect(database=":memory:")

        q_exp = f"""
            SELECT
                CAST({EXPORTER_COL} AS VARCHAR) AS code,
                SUBSTR(CAST({COMMODITY_COL} AS VARCHAR), 1, 3) AS sitc3,
                SUM(CAST({TRADE_VAR} AS DOUBLE)) AS total
            FROM read_parquet('{parquet}')
            GROUP BY 1, 2
        """
        exp = con.execute(q_exp).df()

        q_imp = f"""
            SELECT
                CAST({IMPORTER_COL} AS VARCHAR) AS code,
                SUBSTR(CAST({COMMODITY_COL} AS VARCHAR), 1, 3) AS sitc3,
                SUM(CAST({TRADE_VAR} AS DOUBLE)) AS total
            FROM read_parquet('{parquet}')
            GROUP BY 1, 2
        """
        imp = con.execute(q_imp).df()
        con.close()

        exp["sitc3"] = exp["sitc3"].astype(str).str.zfill(3)
        imp["sitc3"] = imp["sitc3"].astype(str).str.zfill(3)

        sectors = sorted(set(exp["sitc3"]) | set(imp["sitc3"]))

        exp_not_loc = int((~exp["code"].isin(uni)).sum())
        imp_not_loc = int((~imp["code"].isin(uni)).sum())

        for s in sectors:
            exp_vec = (
                exp.loc[exp["sitc3"] == s, ["code", "total"]]
                .set_index("code")["total"]
                .reindex(uni)
                .fillna(0.0)
                .values
            )

            imp_vec = (
                imp.loc[imp["sitc3"] == s, ["code", "total"]]
                .set_index("code")["total"]
                .reindex(uni)
                .fillna(0.0)
                .values
            )

            y_exp = log10_1p(exp_vec)
            y_imp = log10_1p(imp_vec)

            rows.append({
                "year": year,
                "sitc3": s,
                "flow": "exports",
                "n": n,
                **moran_normal_report(y_exp, w),
                "zero_share": float((exp_vec == 0).mean()),
                "exp_not_locatable_codes": exp_not_loc,
                "imp_not_locatable_codes": imp_not_loc,
            })

            rows.append({
                "year": year,
                "sitc3": s,
                "flow": "imports",
                "n": n,
                **moran_normal_report(y_imp, w),
                "zero_share": float((imp_vec == 0).mean()),
                "exp_not_locatable_codes": exp_not_loc,
                "imp_not_locatable_codes": imp_not_loc,
            })

        if year % 5 == 0:
            print(f"Processed {year}")

    out = pd.DataFrame(rows).sort_values(["year", "sitc3", "flow"])
    out.to_csv(OUT_CSV, index=False)

    print(f"Saved: {OUT_CSV}")
    print(out.head(10).to_string(index=False))


if __name__ == "__main__":
    main()


Processed 1980
Processed 1985
Processed 1990
Processed 1995
Processed 2000
Processed 2005
Processed 2010
Processed 2015
Processed 2020
Saved: C:\Python\trade\dataverse_files\outputs_moran\moran_sitc3_S2_1976_2023_normal_inference.csv
 year sitc3    flow   n  moran_i   mean_H0  variance_H0        z  p_value      ci_5    ci_95  zero_share  exp_not_locatable_codes  imp_not_locatable_codes
 1976   001 exports 252 0.038639 -0.003984     0.000671 1.645641 0.099838 -0.012126 0.089404    0.464286                        0                        0
 1976   001 imports 252 0.032302 -0.003984     0.000671 1.400994 0.161216 -0.018463 0.083067    0.333333                        0                        0
 1976   011 exports 252 0.023122 -0.003984     0.000671 1.046554 0.295305 -0.027643 0.073887    0.436508                        0                        0
 1976   011 imports 252 0.003314 -0.003984     0.000671 0.281769 0.778121 -0.047451 0.054079    0.313492                        0                 

**HeatMap 3 Digit**

In [30]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ============================================================
# CONFIG
# ============================================================
MORAN_SITC3_CSV = r"C:\Python\trade\dataverse_files\outputs_moran\moran_sitc3_S2_1976_2023_normal_inference.csv"

CODES_DIR = r"C:\Python\trade\dataverse_files\S02 CODES"
SITC3_DICT_PATH = os.path.join(CODES_DIR, "sitc2-3digit.txt")  # must contain 3-digit SITC codes

OUT_DIR = os.path.dirname(MORAN_SITC3_CSV)
OUT_EXPORTS = os.path.join(OUT_DIR, "heatmap_moran_sitc3_level_exports.png")
OUT_IMPORTS = os.path.join(OUT_DIR, "heatmap_moran_sitc3_level_imports.png")

# Fixed diverging scale requested
VMIN, VCENTER, VMAX = -0.5, 0.0, 0.5


# ============================================================
# ROBUST TXT PARSER (delimiter-agnostic)
# ============================================================
def read_code_dictionary(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = [ln.strip() for ln in f if ln.strip() and not ln.strip().startswith("#")]

    candidates = ["\t", "|", ";", ","]
    best = None

    def try_parse(delim: str):
        rows = [ln.split(delim) for ln in lines]
        k = max(len(r) for r in rows)
        if k < 2:
            return None
        rows = [r + [""] * (k - len(r)) for r in rows]
        return pd.DataFrame(rows)

    for d in candidates:
        df = try_parse(d)
        if df is not None and df.shape[1] >= 3:
            best = df
            break

    if best is None:
        rows = [re.split(r"\s{2,}", ln) for ln in lines]
        k = max(len(r) for r in rows)
        rows = [r + [""] * (k - len(r)) for r in rows]
        best = pd.DataFrame(rows)

    df = best.iloc[:, :3].copy()
    df.columns = ["code", "name", "nickname"]

    # pandas 3.x compatible stripping
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip()

    # Drop header-like first row if code isn't numeric-ish
    if len(df):
        first_code = str(df.loc[0, "code"]).strip()
        if not re.fullmatch(r"\d{1,4}", first_code or ""):
            df = df.iloc[1:].reset_index(drop=True)

    return df


def build_sitc3_labels(dict_df: pd.DataFrame) -> pd.DataFrame:
    dict_df = dict_df.copy()

    # Keep numeric part only; preserve leading zeros via zfill(3)
    dict_df["code"] = dict_df["code"].str.extract(r"(\d+)")[0].fillna(dict_df["code"])
    dict_df["code"] = dict_df["code"].astype(str).str.zfill(3)

    # Validate 3-digit
    lens = dict_df["code"].str.len().value_counts().to_dict()
    if set(lens.keys()) - {3}:
        raise ValueError(
            f"Dictionary codes are not strictly 3-digit after zfill. Length distribution: {lens}. "
            f"Check SITC3_DICT_PATH points to the 3-digit file."
        )

    # Display label: "000 – Nickname"
    dict_df["display"] = dict_df["nickname"].replace({"": np.nan})
    dict_df["display"] = dict_df["display"].fillna(dict_df["name"].replace({"": np.nan}))
    dict_df["display"] = dict_df["display"].fillna(dict_df["code"])
    dict_df["display"] = dict_df["code"] + " – " + dict_df["display"]

    dict_df = dict_df.drop_duplicates(subset=["code"], keep="first")
    return dict_df[["code", "display"]]


# ============================================================
# HEATMAP PLOTTER
# ============================================================
def plot_sitc3_heatmap(moran_csv: str, labels_df: pd.DataFrame, flow: str, out_path: str):
    df = pd.read_csv(moran_csv, dtype={"sitc3": str, "flow": str})
    required = {"year", "flow", "sitc3", "moran_i"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Moran SITC-3 CSV missing columns: {missing}")

    df["sitc3"] = df["sitc3"].astype(str).str.zfill(3)
    df["year"] = pd.to_numeric(df["year"], errors="raise")

    d = df[df["flow"].str.lower() == flow.lower()].copy()
    if d.empty:
        raise ValueError(f"No rows found for flow='{flow}'")

    # Pivot to matrix
    mat = d.pivot(index="sitc3", columns="year", values="moran_i")
    mat = mat.sort_index()
    mat = mat.reindex(sorted(mat.columns), axis=1)

    # Labels
    label_map = dict(zip(labels_df["code"], labels_df["display"]))
    y_labels = [label_map.get(code, code) for code in mat.index]

    # FIXED SCALE (-0.5 to 0.5) + keep arr defined
    arr = mat.values.astype(float)
    norm = TwoSlopeNorm(vmin=-0.20, vcenter=0.0, vmax=0.20)

    # Plot (no title)
    fig, ax = plt.subplots(figsize=(20, 14), dpi=200)
    im = ax.imshow(arr, aspect="auto", interpolation="nearest", norm=norm, cmap="RdBu_r")

    # X axis (years) with reduced tick density
    years = mat.columns.tolist()
    ax.set_xticks(range(len(years)))
    show_every = 2
    ax.set_xticklabels([str(int(y)) if (i % show_every == 0) else "" for i, y in enumerate(years)], rotation=90)

    # Y axis (sectors)
    ax.set_yticks(range(len(y_labels)))
    ax.set_yticklabels(y_labels, fontsize=4)

    ax.set_xlabel("")
    ax.set_ylabel("")

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Moran's I")

    plt.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# RUN
# ============================================================
dict_raw = read_code_dictionary(SITC3_DICT_PATH)
labels = build_sitc3_labels(dict_raw)

plot_sitc3_heatmap(MORAN_SITC3_CSV, labels, flow="exports", out_path=OUT_EXPORTS)
plot_sitc3_heatmap(MORAN_SITC3_CSV, labels, flow="imports", out_path=OUT_IMPORTS)

print("Saved:")
print(OUT_EXPORTS)
print(OUT_IMPORTS)


Saved:
C:\Python\trade\dataverse_files\outputs_moran\heatmap_moran_sitc3_level_exports.png
C:\Python\trade\dataverse_files\outputs_moran\heatmap_moran_sitc3_level_imports.png


**AUDIT**

In [31]:
import os
import json
import numpy as np
import pandas as pd
import duckdb

# ============================
# PATHS
# ============================
BASE_GEO = r"C:\Python\trade\geo"
OD_PATH = os.path.join(BASE_GEO, "OD_Matrix.csv")

BASE_TRADE = r"C:\Python\trade\dataverse_files"
GLOBAL_MORAN = os.path.join(BASE_TRADE, "outputs_moran",
                            "moran_global_S2_1976_2023_normal_inference.csv")
SITC2_MORAN = os.path.join(BASE_TRADE, "outputs_moran",
                            "moran_sitc2_S2_1976_2023_normal_inference.csv")

YEAR_START, YEAR_END = 1976, 2023

# ============================
# LOAD GEOGRAPHY
# ============================
od = pd.read_csv(OD_PATH)
U_OD = set(od["origin"].astype(str)) | set(od["destination"].astype(str))
n = len(U_OD)

# ============================
# 1) CODE MATCHING & DROPPING
# ============================
drop_rows = []

for year in range(YEAR_START, YEAR_END + 1):
    path = os.path.join(BASE_TRADE, f"S2_{year}.parquet")
    con = duckdb.connect(database=":memory:")

    df = con.execute(f"""
        SELECT exporter, importer, value_final
        FROM read_parquet('{path}')
    """).df()
    con.close()

    total_value = df["value_final"].sum()

    valid_exp = df[df["exporter"].isin(U_OD)]
    valid_imp = df[df["importer"].isin(U_OD)]

    captured_exports = valid_exp["value_final"].sum()
    captured_imports = valid_imp["value_final"].sum()

    drop_rows.append({
        "year": year,
        "drop_share_exports": 1 - captured_exports / total_value,
        "drop_share_imports": 1 - captured_imports / total_value,
    })

drop_df = pd.DataFrame(drop_rows)

# ============================
# 2) AGGREGATION IDENTITY
# ============================
glob = pd.read_csv(GLOBAL_MORAN)
sitc2 = pd.read_csv(SITC2_MORAN)

identity_rows = []
for year in range(YEAR_START, YEAR_END + 1):
    g_exp = glob[(glob.year == year) & (glob.flow == "exports")]["n"].iloc[0]
    s_exp = sitc2[(sitc2.year == year) & (sitc2.flow == "exports")].shape[0]

    identity_rows.append({
        "year": year,
        "n_global": g_exp,
        "n_sectoral_rows": s_exp
    })

identity_df = pd.DataFrame(identity_rows)

# ============================
# 3) MORAN PARAMETER CHECKS
# ============================
def moran_checks(df):
    EI_expected = -1 / (n - 1)
    return {
        "EI_deviation_max": float(np.max(np.abs(df["mean_H0"] - EI_expected))),
        "variance_nonpositive": int((df["variance_H0"] <= 0).sum()),
        "out_of_bounds_I": int((np.abs(df["moran_i"]) > 1.05).sum()),
    }

checks = {
    "global": moran_checks(glob),
    "sitc2": moran_checks(sitc2),
}

# ============================
# SAVE AUDIT OUTPUTS
# ============================
audit_summary = {
    "n_spatial_units": n,
    "drop_share_exports_max": float(drop_df["drop_share_exports"].max()),
    "drop_share_imports_max": float(drop_df["drop_share_imports"].max()),
    "moran_checks": checks,
}

out_dir = os.path.join(BASE_TRADE, "outputs_moran", "audit")
os.makedirs(out_dir, exist_ok=True)

drop_df.to_csv(os.path.join(out_dir, "audit_drop_shares.csv"), index=False)
identity_df.to_csv(os.path.join(out_dir, "audit_identity.csv"), index=False)

with open(os.path.join(out_dir, "audit_summary.json"), "w") as f:
    json.dump(audit_summary, f, indent=2)

print("Audit completed. See outputs in:", out_dir)
print(json.dumps(audit_summary, indent=2))


Audit completed. See outputs in: C:\Python\trade\dataverse_files\outputs_moran\audit
{
  "n_spatial_units": 252,
  "drop_share_exports_max": 0.0,
  "drop_share_imports_max": 0.0,
  "moran_checks": {
    "global": {
      "EI_deviation_max": 1.9949319973733282e-17,
      "variance_nonpositive": 0,
      "out_of_bounds_I": 0
    },
    "sitc2": {
      "EI_deviation_max": 1.9949319973733282e-17,
      "variance_nonpositive": 0,
      "out_of_bounds_I": 0
    }
  }
}
